## Set up Gurobipy
---

In Python, you can use `Gurobipy` as an optimization modeling library, which serves as an interface between the user and the solver. `Gurobipy` allows you to formulate and define optimization problems in a high-level, mathematical manner.

The solver, such as `Gurobi`, is responsible for solving the problem and providing an optimal solution.

Additionally, you can utilize `Matplotlib` to visualize the solution and `NumPy` for mathematical calculations.

The installation works with the following commands. `%` can be used to execute shell commands within a Jupyter Notebook code cell. Alternatively, you could execute `pip install gurobipy` and `pip install matplotlib` in your terminal/shell.

`pip` is the package manager for python.

In [ ]:
# First you can check, whether the required packages are already installed
# (if not, you'll receive a warning)
%pip show gurobipy
%pip show matplotlib
%pip show numpy
%pip show pandas

In [ ]:
#%pip install gurobipy
#%pip install matplotlib
#%pip install numpy
#%pip install pandas

Import them into this notebook

In [ ]:
from gurobipy import * # imports everything from gurobipy without alias
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Production Planning

### Decision variables:

Determination of the optimal **production quantities** of each product,

### objective function:
0) **contribution margin maximization**: $ \qquad \max db = \displaystyle\sum_{i=1}^I(e_i - k_i^v)\cdot X_i $



### Constraints:


1) **capacity constraint:** $\hspace{47mm} \sum_{i=1}^I(r_{ij}\cdot X_i) \leq c_j \hspace{31mm} \forall j \in J $

2) **Sales cap:**     $ \hspace{60mm} X_i \leq d_i \hspace{49mm} \forall i \in I $

3) **Non-negativity condition:** $ \hspace{38mm} X_i \geq 0 \hspace{50mm}    \forall i \in I $


* * * 

## Symbols used

### Sets

$i \in (1,..,I) \hspace{20mm}$ Products

$j \in (1,..,J) \hspace{20mm}$ Resources




### Variables

$X_i$     $\geq0 \hspace{30mm}$ Production quantity





### Parameters 

$e_i \hspace{39mm}$   revenue from product i  

$k_i^v \hspace{38mm}$  variable costs for the manufacture of product i 

$d_i \hspace{39mm}$  sales cap of product i   

$r_{ij} \hspace{38mm}$  production coefficient of product i regarding resource j  

$c_j \hspace{39mm}$  capacity of resource  j


* * *


## Create the model named ``m`` and specify Clp as the solver to use.
---

In [ ]:
m = Model()

# create the model m
# Gurobi is used as the solver to solve the model

### Sets and Parameters

Add the Sets

In [ ]:
#Sets

Products = ["Normal", "Grande"]
Resources = ["Wood", "Screws", "Hinges"]

# If the elements in the sets are words, they are written as strings

I = len(Products)
J = len(Resources)

# We determine the number of elements in the sets using the len function
# This information is required for creating variables, constraints etc.

Here we define one-dimensional and two-dimensional parameters

In [ ]:
e  = np.array([30, 40])          # revenue from product i
kv = np.array([20, 25])          # variable production costs of product i
d  = np.array([100, 100])        # demand of product i
c =  np.array([2400, 4000, 900]) # capacity of resource j

               # wood    screws    hinges
r  = np.array([[  6,       20,       4  ], # Normal
               [  20,      20,       9  ]]); # Grande

# production coefficient of product i regarding resource j

# for this small-scale example, using Python's built-in data types (like lists) is simpler and more straightforward
# when we use large datasets of parameters or need to calculate mathematical operations on them, we recommend using NumPy

### Decision variable
---
Define the decision variable. Pay attention to the definition range (constraint 3).

3) **Non-negativity constraint**: Since no negative quantities of the products can be produced, an additional non-negativity constraint is introduced.

 $ \qquad X_i \geq 0 \qquad \qquad \qquad    \forall i \in I $

In [ ]:
X = m.addVars(I, lb=0, name="X")

# addVars creates a new variables with the name X
# m.addVars indicates that these variables are associated with the model m
# The variables have an index which counts from 0 to I-1 (in this case I=2)
# The lower bound of the variable is set to 0 with the lb parameter

- - -
 
 ## Objective Function 

0) **contribution margin maximization**: The total contribution margin db is to be maximized. This is calculated from the sum of the individual product contribution margins (product revenue less variable costs) multiplied by the corresponding product quantities of the different goods.

$ \qquad \max db = \displaystyle\sum_{i=1}^I(e_i - k_i^v)\cdot X_i $ 


 

In [ ]:
m.setObjective(sum((e[i] - kv[i]) * X[i] for i in range(I)), GRB.MAXIMIZE)

# The setObjective function sets the objective function for the model m
# We form a sum with the sum function
# The range over which the sum is calculated is defined after the for keyword
# Here, all products i are summed from 0 to I-1

- - -

## Constraints



1) **capacity constraint**: Possibly there are capacity restrictions of the resource j for the production time possible on them. The sum of the total production time of all products i=1, ..., I on each resource j may not exceed the available capacity $ c_j $ .


$\qquad \sum_{i=1}^I(r_{ij}\cdot X_i) \leq c_j \qquad \forall j \in J $



In [ ]:
m.addConstrs((sum(r[i][j] * X[i] for i in range(I)) <= c[j] for j in range(J)), name="KapRes");

# Constraints are created with the addConstrs function
# Constraints are named, in this case KapRes
# This expression is creating a constraint for each resource j. 
# Each constraint itself is a sum of terms over all products i. 
# So, in total, it will generate J constraints, and each constraint contains I terms.

2) **Sales cap**: It is possible that upper sales limits exist for the products based on demand. The produced quantity of each product i may not exceed this sales cap.
 
 $ \qquad X_i \leq d_i \qquad \qquad \qquad \forall i \in I $


In [ ]:
m.addConstrs((X[i] <= d[i] for i in range(I)), name="AbsOb");

## Solve the model.
---

In [ ]:
m.optimize()

# The optimize function solves the created model m

Display the complete model

In [ ]:
m.display()

# the display() function will return the whole model again

In [ ]:
m.printStats()

# the printStats() function will also give you a small summary

Display the objective value

In [ ]:
db = m.objVal

print("Objective value db: ", round(db))

# We can display the objective value of the model with m.objVal
# Here, the objective value is assigned to the parameter db
# We round values to whole numbers with the round function

Display the production quantities

In [ ]:
m.printAttr('X')

In [ ]:
X_vals = np.array([[X[i].x for i in range(I)]])

# We also can store the values in a matrix like this.
# Values of variables can be displayed by accessing the 'x' attribute of the variable objects
# We use a matrix np.array([[...]]) instead of an array np.array([...]) to be able to transpose it later

print(X_vals)

Plot the resource consumption compared to the available capacity

In [ ]:
X_vals = X_vals.T # transpose X_vals and make it multiplicable with 'r'.

bardata = np.multiply(X_vals, r).T

print(bardata)

In [ ]:
# Extracting the first and second columns of data from 'bardata' for bar heights.
column1 = bardata[:,0]
column2 = bardata[:,1]

# Creating the first set of bars in the bar chart using data from 'column1' and coloring them 'steelblue'.
plt.bar(Resources, column1, color='steelblue')

# Creating the second set of bars, positioned above the first set ('bottom=column1') using data from 'column2' and coloring them 'lightskyblue'.
plt.bar(Resources, column2, bottom=column1, color='lightskyblue')

# Creating red horizontal line markers at positions specified by 'c' across 'Resources'.
plt.scatter(Resources, c, color='r', marker="_", s=400, label='data 1')

# Labeling y and x axis
plt.xlabel('Resources')
plt.ylabel('Capacity')

# Adding a title 'Capacity per resource' to the plot.
plt.title('Capacity per resource')

# Creating a legend for the plot, the red markers are labeled 'data 1' and it seems like there's an attempt to label bars using 'Products'.
plt.legend(["Capacity"] + Products)

# Displaying the plot.
plt.show()
